In [19]:
import cv2
vid = cv2.VideoCapture("badapple.mp4")
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = vid.get(cv2.CAP_PROP_FPS)
frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

def print_bw(text, white, **kwargs):
    if (white):
        print(f"\x1b[37m{text}\x1b[0m", **kwargs)
    else:
        print(f"\x1b[30m{text}\x1b[0m", **kwargs)

class VideoSerializer():
    def __init__(self, f):
        self.prev = 0
        self.ctr = 0
        self.total_bytes = 0
        self.f = f
    def feed(self, image):
        flat_img = image.flatten()
        for x in flat_img:
                if self.prev != x:
                    #print_bw(f"{self.ctr:02x}", self.prev, end="")
                    self.f.write(bytearray([self.ctr]))
                    self.total_bytes+=1
                    self.ctr=1
                    self.prev = x
                else:
                    self.ctr+=1
                    if self.ctr==255:
                        #print_bw(f"{self.ctr:02x}", self.prev, end="")
                        self.f.write(bytearray([self.ctr]))
                        self.total_bytes+=1
                        self.ctr=0
                        self.prev=255-self.prev

    def finish(self):
        #print_bw(f"{self.ctr:02x}", self.prev, end="")
        self.f.write(bytearray([self.ctr]))
        self.total_bytes+=1
        print(f'{self.total_bytes=}')

test_image=None


print(width, height, fps, frames)
f = open("badapple.bin", "wb")
ser = VideoSerializer(f)
for i in range(frames):
    success, image = vid.read()
    image = cv2.resize(image, (180, 135))
    image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    success, image = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
    if i==1900:
        test_image=image
    ser.feed(image)
ser.finish()
f.close()

1444 1080 29.97002997002997 6955
self.total_bytes=3021337


In [ ]:
#3020765
#3021338
#3021338
total_bytes, frames*135*180/8

(3020765, 21125812.5)

In [ ]:
flat_img = image.flatten()
for i in range(0,len(flat_img), 4):
    print(f'{flat_img[i]*8|flat_img[i+1]*4|flat_img[i+2]*2+flat_img[i+3]:x}', end='')

0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

/tmp/ipykernel_24036/2178021363.py:6: RuntimeWarning: overflow encountered in scalar multiply
  print(f'{flat_img[i]*8|flat_img[i+1]*4|flat_img[i+2]*2+flat_img[i+3]:x}', end='')
/tmp/ipykernel_24036/2178021363.py:6: RuntimeWarning: overflow encountered in scalar add
  print(f'{flat_img[i]*8|flat_img[i+1]*4|flat_img[i+2]*2+flat_img[i+3]:x}', end='')


In [ ]:
compress_image(test_image)

ff00b504af05ff00ff000201ff00ff008e01ff00ff00ff001401160301040406a1165401481a9a1b981c981d971d961f020113017d2490248e268d282b0160288b2a8a2c882b892a8a2b882c0a017d2d0801802b040102010e0102016f2b04012a011201452c06011102070161310501120107016033040103010e0107015f36010319010201543f01010f01030202010201040104034a4213020301010108024c4304010c020502070101014d430401070105010102010204035043050103010201050103010301020250450801080102020204020112023b450501050106035847090107020c010101484a080107030c0101010201444a0301030207020601030103010201454f0202070205010303030205013f510102070202010101050201040401414f02050402030107050201010235010c4d0105010102040301080201020103040330010b520101020502010a030103010142510101010101090902010305020d013151040301020e01010209010b01314f0504010101020b051501314e060202020e07110101012c01054c060303010e02070101010c010101324d040303020c0317010101325305010c03190132520401010209030301020105010d020e01010106011a510602010105040401010106010b0102010f02225002010401010103050302020106010c0102010a010301

1206

In [ ]:
def compress_image(image):
    bytes = 0
    flat_img = image.flatten()
    # bl wh bl wh ...
    prev = 0
    ctr=0
    for x in flat_img:
        if prev != x:
            print_bw(f"{ctr:02x}", prev, end="")
            bytes+=1
            ctr=1
            prev = x
        elif ctr==255:
            print_bw(f"{ctr:02x}", prev, end="")
            print_bw("00", 255-prev, end="")
            bytes+=2
            ctr = 1
            prev = x
        else:
            ctr+=1
    return bytes

In [20]:
import cv2
import numpy as np

# Dimensions as defined in the encoder script
WIDTH = 180
HEIGHT = 135
FRAME_SIZE = WIDTH * HEIGHT
FPS = 30.0 # Assuming standard 30 FPS, adjust if your source differed

def reconstruct_video(bin_path="badapple.bin", out_path="reconstructed.mp4"):
    # 1. Read the serialized binary run-length data
    print("Reading binary file...")
    with open(bin_path, "rb") as f:
        # Read the file directly into a numpy array of 8-bit unsigned integers
        run_lengths = np.frombuffer(f.read(), dtype=np.uint8)

    # 2. Reconstruct the color sequence
    # The encoder always starts with prev=0 (black) and alternates on every new run length.
    # We create an array of [0, 255, 0, 255, 0, 255...] to match the lengths.
    colors = np.zeros(len(run_lengths), dtype=np.uint8)
    colors[1::2] = 255 

    # 3. Decode the RLE
    # np.repeat takes an array of items and an array of counts, expanding them.
    # e.g., np.repeat([0, 255], [3, 2]) becomes [0, 0, 0, 255, 255]
    print("Decoding pixel data (this may take a moment)...")
    flat_pixels = np.repeat(colors, run_lengths)

    # 4. Group the flat pixels back into frames
    num_frames = len(flat_pixels) // FRAME_SIZE
    print(f"Successfully recovered {num_frames} frames.")

    # Drop any trailing pixels at the very end that don't complete a full frame
    flat_pixels = flat_pixels[:num_frames * FRAME_SIZE]

    # Reshape the 1D array into a 3D array: (frame_index, y, x)
    frames = flat_pixels.reshape((num_frames, HEIGHT, WIDTH))

    # 5. Write the reconstructed frames to a video file
    print("Writing to video file...")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    
    # isColor=False is important here since our frames only have 1 channel (grayscale)
    out = cv2.VideoWriter(out_path, fourcc, FPS, (WIDTH, HEIGHT), isColor=False)

    for frame in frames:
        out.write(frame)

    out.release()
    print(f"Done! Saved to {out_path}")

if __name__ == "__main__":
    reconstruct_video()

Reading binary file...
Decoding pixel data (this may take a moment)...
Successfully recovered 6955 frames.
Writing to video file...
Done! Saved to reconstructed.mp4
